# Human vs Machine Translation Detection - Inference Pipeline

## Overview
This notebook provides an inference pipeline for detecting whether translations are human-generated (HT) or machine-generated (MT) using a fine-tuned transformer model.

## Workflow

### 1. Model & Data Loading
- **load_model()**: Loads a fine-tuned AutoModel and tokenizer from disk, automatically detecting GPU/CPU availability.
- **Input**: Model path, device specification (optional)
- **Output**: Model, tokenizer, and device identifier

### 2. Inference Methods
Two inference approaches are provided:

- **predict_single()**: Processes individual text samples
    - Returns predicted label (HT/MT) and confidence scores for both classes
    
- **predict_batch()**: Efficiently processes multiple texts in batches
    - Reduces memory usage and improves throughput
    - Configurable batch size and max token length

### 3. CSV Processing Pipeline
- **run_model_on_csv_column()**: End-to-end inference on CSV data
    - Reads input CSV and extracts text from specified column(s)
    - Runs batch inference on all texts
    - Appends predictions with confidence scores to original data
    - Saves enhanced CSV with model-specific column naming

### 4. Batch Processing Loop
- Processes multiple columns (different translation sources: Claude, Gemini, Llama, NLLB, OpenAI, Qwen, etc.)
- Generates separate output files for each column
- Uses consistent model parameters across all runs

## Configuration
Before running, set:
- `MODEL_PATH`: Path to your fine-tuned HT/MT detection model
- `INPUT_CSV`: Path to your CSV with translations
- `MODEL_NAME`: Short identifier for model (used in output filenames)
- `columns`: List of CSV columns to analyze

## Output
For each column processed, generates a CSV with:
- Original data columns
- Predicted labels (HT/MT)
- Confidence scores for human and machine translation
- Confidence difference metric

In [ ]:
import torch
import pandas as pd
import csv
import os
import re
from typing import List, Dict

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
def load_model(model_path: str, device: str = None):
    """
    Load fine-tuned model and tokenizer from disk.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading model from: {model_path}")
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    model.to(device)
    model.eval()

    return model, tokenizer, device

In [ ]:
def predict_single(
    model,
    tokenizer,
    text: str,
    device: str,
    max_length: int = 512
) -> Dict:
    """
    Run inference on a single text input.
    Returns label + confidence scores.
    """

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

    confidence_ht = float(probs[0])
    confidence_mt = float(probs[1])

    label = "MT" if confidence_mt > confidence_ht else "HT"

    return {
        "predicted_label": label,
        "confidence_ht": confidence_ht,
        "confidence_mt": confidence_mt,
        "confidence_diff": abs(confidence_mt - confidence_ht)
    }

In [ ]:
def predict_batch(
    model,
    tokenizer,
    texts: List[str],
    device: str,
    batch_size: int = 16,
    max_length: int = 512
) -> List[Dict]:

    results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()

        for p in probs:
            confidence_ht = float(p[0])
            confidence_mt = float(p[1])
            label = "MT" if confidence_mt > confidence_ht else "HT"

            results.append({
                "predicted_label": label,
                "confidence_ht": confidence_ht,
                "confidence_mt": confidence_mt,
                "confidence_diff": abs(confidence_mt - confidence_ht)
            })

    return results

In [ ]:
def run_model_on_csv_column(
    model_path: str,
    input_csv_path: str,
    text_column: str,
    output_csv_path: str,
    batch_size: int = 16,
    max_length: int = 512
) -> pd.DataFrame:

    # Load model
    model, tokenizer, device = load_model(model_path)

    # Load CSV
    df = pd.read_csv(input_csv_path)

    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found in CSV.")

    texts = df[text_column].fillna("").astype(str).tolist()

    print(f"Running inference on {len(texts)} rows...")

    results = predict_batch(
        model=model,
        tokenizer=tokenizer,
        texts=texts,
        device=device,
        batch_size=batch_size,
        max_length=max_length
    )

    pred_df = pd.DataFrame(results)

    model_name = os.path.basename(model_path.rstrip("/"))
    safe_model_name = re.sub(r"[^a-zA-Z0-9_]+", "_", model_name.lower())

    pred_df = pred_df.rename(columns={
        col: f"{safe_model_name}_{col}" for col in pred_df.columns
    })

    output_df = pd.concat([df, pred_df], axis=1)
    output_df.to_csv(output_csv_path, index=False, encoding="utf-8")

    print(f"Saved output to: {output_csv_path}")

    return output_df

In [ ]:
MODEL_PATH = "" # Set this to your fine-tuned model path, e.g. "models/mdeberta_300_epoch3". Make sure the model is fine-tuned for HT vs MT detection.

INPUT_CSV = "" # Set this to your input CSV file path, which should have the translations you want to analyze.

columns=["human translation","claude","gemini","gemma","llamma","nllb","openai","qwen"]
for col in columns:
    TEXT_COLUMN = col
    MODEL_NAME="" # Set this to a short name for your model, e.g. "mdeberta_300". This will be used in the output CSV filename.
    OUTPUT_CSV = f"model_{MODEL_NAME}_col_{TEXT_COLUMN}_mdeb_chunk.csv"


    df_out = run_model_on_csv_column(
        model_path=MODEL_PATH,
        input_csv_path=INPUT_CSV,
        text_column=TEXT_COLUMN,
        output_csv_path=OUTPUT_CSV,
        batch_size=16,
        max_length=512
    )
